# Part 4 — Generative AI: Analyse Patient Follow-Up Notes

In this section, Generative AI is used to analyse patient follow-up notes and identify possible reasons for missed appointments.

The workflow includes structured output, Pydantic validation, prompt testing, error handling, and combining the AI-generated insights with appointment data.

## 4.1 — Loading and Exploring Patient Follow-Up Notes

In this section, the patient follow-up notes dataset is loaded and explored.

The dataset is inspected to understand its structure and the information recorded in each patient note. An individual note is also examined to understand the type of information that will be analysed using Generative AI.

In [ ]:
import pandas as pd

notes_df = pd.read_csv('/content/followup_notes.csv')

print("Dataset shape:", notes_df.shape)
print("\nFirst 5 notes:")
display(notes_df.head())

# Extract the first note
n001 = notes_df.iloc[0]

print("\nNote ID:", n001['note_id'])
print("Appointment ID:", n001['appointment_id'])
print("Note text:", n001['note_text'])

Dataset shape: (30, 3)

First 5 notes:


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."



Note ID: N001
Appointment ID: A0872
Note text: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.


In [ ]:
!pip install -q google-genai pydantic

In [ ]:
import json
import time
import pandas as pd

from getpass import getpass
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

API_KEY = getpass("Paste your key : ").strip()
client= genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-3.5-flash"

Paste your key : ··········


In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {
            "type": "string"
        },
        "confidence": {
            "type": "number"
        },
        "summary": {
            "type": "string"
        },
        "recommended_action": {
            "type": "string"
        }
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

print("JSON schema created successfully!")

JSON schema created successfully!


In [ ]:
ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

print("Allowed root causes:")
print(ALLOWED_CAUSES)

Allowed root causes:
['forgot_or_no_reminder', 'distance_or_transport', 'cost_or_fee', 'illness_or_emergency', 'work_or_schedule', 'other']


In [ ]:
SYSTEM_TEXT = """
You are an assistant helping a clinic understand why patients miss appointments.

Classify patient follow-up notes accurately.
Use only the information present in the note.
Do not invent reasons.
If the reason is unclear, choose other.

Return valid JSON with exactly four fields:
root_cause, confidence, summary, recommended_action.

The root_cause must be one of:
forgot_or_no_reminder,
distance_or_transport,
cost_or_fee,
illness_or_emergency,
work_or_schedule,
other.

Confidence must be between 0 and 1.
Summary must contain no more than 25 words.
"""


def analyse_note(note_text):

    prompt = f"""
    CONTEXT:
    A clinic is analysing reasons for missed appointments.

    ROLE:
    You are a careful patient follow-up analyst.

    ACTION:
    Analyse this patient note and identify the most likely root cause.
    Suggest one practical action the clinic can take.

    TONE:
    Professional, clear, and concise.

    FORMAT:
    Return only JSON with exactly these four fields:
    root_cause, confidence, summary, recommended_action.

    Patient note:
    {note_text}
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    data = json.loads(response.text)

    return data


print("analyse_note function created successfully!")

analyse_note function created successfully!


In [ ]:
result_n001 = analyse_note(n001["note_text"])

print(result_n001)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient missed the appointment because she forgot and did not receive a reminder message.', 'recommended_action': 'Verify patient contact details and send automated SMS reminders prior to appointments.'}


## 4.2 — Structured Output and Pydantic Validation

In this section, a structured output schema is defined to guide the Generative AI model.

Pydantic is used to validate the AI-generated responses and ensure that:
- The root cause belongs to the permitted categories.
- Confidence scores remain between 0 and 1.
- The summary and recommended action contain valid text.

Blank patient notes are handled separately rather than forcing the AI model to guess a reason for the missed appointment.

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

print("Pydantic validation model created!")

Pydantic validation model created!


In [ ]:
import pandas as pd

appointments = pd.read_csv("/content/appointments_clean.csv")
clinics = pd.read_csv("/content/clinics_enriched.csv")
notes = pd.read_csv("/content/followup_notes.csv")

print("Appointments:", appointments.shape)
print("Clinics:", clinics.shape)
print("Follow-up notes:", notes.shape)

Appointments: (990, 16)
Clinics: (8, 7)
Follow-up notes: (30, 3)


In [ ]:
print("Gemini client:", "Ready" if "client" in globals() else "Missing")
print("analyse_note:", "Ready" if "analyse_note" in globals() else "Missing")
print("JSON schema:", "Ready" if "SCHEMA" in globals() else "Missing")
print("Pydantic model:", "Ready" if "FollowupInsight" in globals() else "Missing")

Gemini client: Ready
analyse_note: Ready
JSON schema: Ready
Pydantic model: Ready


In [ ]:
def analyse_note_with_raw(note_text):

    prompt = f"""
    CONTEXT:
    A clinic is analysing reasons for missed appointments.

    ROLE:
    You are a careful patient follow-up analyst.

    ACTION:
    Analyse this patient note and identify the most likely root cause.
    Suggest one practical action the clinic can take.

    TONE:
    Professional, clear, and concise.

    FORMAT:
    Return only JSON with exactly these four fields:
    root_cause, confidence, summary, recommended_action.

    Use only these root causes:
    forgot_or_no_reminder,
    distance_or_transport,
    cost_or_fee,
    illness_or_emergency,
    work_or_schedule,
    other.

    If unclear, choose other. Do not guess.

    Summary must contain no more than 25 words.
    Confidence must be between 0 and 1.

    Patient note:
    {note_text}
    """

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    raw_text = response.text
    data = json.loads(raw_text)

    return data, raw_text


print("Raw-response helper created successfully!")

Raw-response helper created successfully!


In [ ]:
notes_df = pd.read_csv("/content/followup_notes.csv")

print("First 10 notes:")
display(notes_df.head(10))

print("Total notes:", len(notes_df))

First 10 notes:


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."
5,N006,A0507,Manager called an urgent meeting at the office...
6,N007,A0333,"Bhool gaya, SMS nahi aaya tha. Dobara booking ..."
7,N008,A0222,"City bus strike that day, no auto available ne..."
8,N009,A0977,"Phone was switched off, could not reach the pa..."
9,N010,A0596,Had a shift change at the factory and could no...


Total notes: 30


In [ ]:
results = []

first_10 = notes_df.head(10)

for index, row in first_10.iterrows():

    note_text = row["note_text"]

    # Create the basic output record
    result = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": "",
        "confidence": None,
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Check missing or blank notes
    if pd.isna(note_text) or str(note_text).strip() == "":

        result["root_cause"] = "other"
        result["confidence"] = 0.0
        result["summary"] = "No note recorded"
        result["recommended_action"] = "Re-call patient"
        result["validation_status"] = "skipped_blank"

        print(row["note_id"], "Skipped: blank note")

    else:

        try:
            data, raw_text = analyse_note_with_raw(note_text)

            result["raw_response"] = raw_text

            # Preserve whatever fields Gemini returned
            if isinstance(data, dict):
                result["root_cause"] = data.get("root_cause", "")
                result["confidence"] = data.get("confidence", None)
                result["summary"] = data.get("summary", "")
                result["recommended_action"] = data.get(
                    "recommended_action", ""
                )

            # Pydantic validation
            validated = FollowupInsight.model_validate(data)

            # Additional allowed-category validation
            if validated.root_cause not in ALLOWED_CAUSES:
                raise ValueError(
                    f"Invalid root cause: {validated.root_cause}"
                )

            result["validation_status"] = "valid"

            print(row["note_id"], "Valid:", validated.root_cause)

        except Exception as e:

            result["validation_status"] = "invalid"

            print(row["note_id"], "Invalid:", str(e))

        # Wait between API calls, but not after the last nonblank note
        remaining_notes = first_10.loc[
            first_10.index > index, "note_text"
        ]

        if any(
            not pd.isna(x) and str(x).strip() != ""
            for x in remaining_notes
        ):
            time.sleep(13)

    results.append(result)


print("\nProcessing completed!")

N001 Valid: forgot_or_no_reminder
N002 Valid: distance_or_transport
N003 Valid: cost_or_fee
N004 Skipped: blank note
N005 Valid: illness_or_emergency
N006 Valid: work_or_schedule
N007 Valid: forgot_or_no_reminder
N008 Valid: distance_or_transport
N009 Valid: other
N010 Valid: work_or_schedule

Processing completed!


In [ ]:
results_df = pd.DataFrame(results)

columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

results_df = results_df[columns]

results_df.to_csv(
    "/content/notes_analysed.csv",
    index=False
)

display(results_df)

print("Saved successfully!")
print("Shape:", results_df.shape)

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_reminder,0.98,The patient completely forgot about the visit ...,Verify patient contact details and send automa...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide..."
1,N002,A0897,distance_or_transport,0.95,Patient missed the appointment because they li...,Offer telehealth options or provide informatio...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
2,N003,A0526,cost_or_fee,1.00,Patient skipped the appointment due to high co...,Provide information on available financial ass...,valid,"{""root_cause"":""cost_or_fee"",""confidence"":1.0,""..."
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,illness_or_emergency,0.98,The patient missed the appointment due to a hi...,Follow up with the patient in a few days to ch...,valid,"{""root_cause"":""illness_or_emergency"",""confiden..."
5,N006,A0507,work_or_schedule,0.95,Patient missed the appointment due to an urgen...,Offer evening or flexible after-hours appointm...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":..."
6,N007,A0333,forgot_or_no_reminder,0.95,The patient forgot the appointment because the...,Verify the patient's contact details and ensur...,valid,"{\n ""root_cause"": ""forgot_or_no_reminder"",\n ..."
7,N008,A0222,distance_or_transport,0.95,Patient missed the appointment due to a city b...,Offer telehealth options or assist with coordi...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
8,N009,A0977,other,0.95,The patient could not be reached after three a...,Try reaching out via an alternative contact me...,valid,"{""root_cause"":""other"",""confidence"":0.95,""summa..."
9,N010,A0596,work_or_schedule,0.95,Patient missed the appointment due to an unexp...,Offer the patient a weekend or Saturday appoin...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":..."


Saved successfully!
Shape: (10, 8)


In [ ]:
# Check final processing results

print("Total records:", len(results_df))

print("\nValidation status:")
print(results_df["validation_status"].value_counts())

print("\nRoot cause distribution:")
print(results_df["root_cause"].value_counts())

print("\nFinal dataset shape:", results_df.shape)

# Save final CSV again
results_df.to_csv(
    "/content/notes_analysed.csv",
    index=False
)

print("\nFinal CSV saved successfully!")

Total records: 10

Validation status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

Root cause distribution:
root_cause
forgot_or_no_reminder    2
distance_or_transport    2
other                    2
work_or_schedule         2
cost_or_fee              1
illness_or_emergency     1
Name: count, dtype: int64

Final dataset shape: (10, 8)

Final CSV saved successfully!


## 4.3 — Break it on purpose


In [ ]:
# Test 1: Invalid confidence score

bad_reply_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment.",
    "recommended_action": "Send SMS reminders."
}

try:
    result = FollowupInsight(**bad_reply_1)
    print("Validation passed")

except ValidationError as e:
    print("Validation failed:")
    print(e)

Validation failed:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [ ]:
# Test 2: Missing recommended_action

bad_reply_2 = {
    "root_cause": "distance_or_transport",
    "confidence": 0.9,
    "summary": "Patient lives far away."
}

try:
    result = FollowupInsight(**bad_reply_2)
    print("Validation passed")

except ValidationError as e:
    print("Validation failed:")
    print(e)

Validation failed:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'distance_...atient lives far away.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:
import json
import time

comparison = []

for note_id in ["N007", "N009"]:

    note_text = notes_df.loc[
        notes_df["note_id"] == note_id, "note_text"
    ].iloc[0]

    # Reuse the existing CRAFT result from Task 4.2
    v2_result = results_df.loc[
        results_df["note_id"] == note_id
    ].iloc[0]

    comparison.append({
        "note_id": note_id,
        "prompt_version": "V2 - CRAFT",
        "root_cause": v2_result["root_cause"],
        "confidence": v2_result["confidence"],
        "summary": v2_result["summary"]
    })

    # Run V1: simple prompt (one API call per note)
    try:
        response = client.models.generate_content(
            model="gemini-3.5-flash",
            contents=f"Classify this note: {note_text}",
            config=types.GenerateContentConfig(
                temperature=0,
                max_output_tokens=1000,
                response_mime_type="application/json",
                response_schema=SCHEMA,
                thinking_config=types.ThinkingConfig(
                    thinking_level="minimal"
                )
            )
        )

        data = json.loads(response.text)

        comparison.append({
            "note_id": note_id,
            "prompt_version": "V1 - Simple",
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", None),
            "summary": data.get("summary", "")
        })

        print(note_id, "V1 completed")

    except Exception as e:
        comparison.append({
            "note_id": note_id,
            "prompt_version": "V1 - Simple",
            "root_cause": "API_ERROR",
            "confidence": None,
            "summary": str(e)
        })

        print(note_id, "V1 error:", str(e))

    if note_id == "N007":
        time.sleep(13)

comparison_df = pd.DataFrame(comparison)

display(comparison_df)

N007 V1 completed
N009 V1 completed


,note_id,prompt_version,root_cause,confidence,summary
0,N007,V2 - CRAFT,forgot_or_no_reminder,0.95,The patient forgot the appointment because the...
1,N007,V1 - Simple,Customer forgot due to missing SMS confirmatio...,0.92,The customer forgot about the service because ...
2,N009,V2 - CRAFT,other,0.95,The patient could not be reached after three a...
3,N009,V1 - Simple,Patient Unreachable,0.95,Unable to reach the patient after three attemp...


In [ ]:

import pandas as pd
import os

# Load the two datasets
appointments_df = pd.read_csv('/content/appointments_clean.csv')
notes_df = pd.read_csv('/content/notes_analysed.csv')

# Merge using appointment_id
merged_df = pd.merge(
    notes_df,
    appointments_df,
    on='appointment_id',
    how='left',
    validate='many_to_one'
)

# Check the merged dataset
print("Merged dataset shape:", merged_df.shape)
print("\nTotal records:", len(merged_df))

# Check whether any appointments failed to match
print(
    "\nUnmatched appointments:",
    merged_df['reminder_sent'].isna().sum()
)

# Display selected columns
display(
    merged_df[
        ['note_id', 'appointment_id', 'root_cause', 'reminder_sent']
    ]
)

# Save merged dataset
merged_df.to_csv('/content/notes_appointments_merged.csv', index=False)

print("\nMerged dataset saved successfully!")


Merged dataset shape: (10, 23)

Total records: 10

Unmatched appointments: 0


,note_id,appointment_id,root_cause,reminder_sent
0,N001,A0872,forgot_or_no_reminder,No
1,N002,A0897,distance_or_transport,No
2,N003,A0526,cost_or_fee,Yes
3,N004,A0926,other,No
4,N005,A0534,illness_or_emergency,Yes
5,N006,A0507,work_or_schedule,Yes
6,N007,A0333,forgot_or_no_reminder,No
7,N008,A0222,distance_or_transport,Yes
8,N009,A0977,other,Yes
9,N010,A0596,work_or_schedule,No



Merged dataset saved successfully!


In [ ]:

# Cross-tabulate root cause against reminder status

reminder_table = pd.crosstab(
    merged_df['root_cause'],
    merged_df['reminder_sent']
)

print("Root Cause vs Reminder Sent:")
display(reminder_table)

# Save the table
reminder_table.to_csv('/content/root_cause_reminder_crosstab.csv')

print("Cross-tabulation saved successfully!")


Root Cause vs Reminder Sent:


reminder_sent,No,Yes
root_cause,,
cost_or_fee,0,1
distance_or_transport,1,1
forgot_or_no_reminder,2,0
illness_or_emergency,0,1
other,1,1
work_or_schedule,1,1


Cross-tabulation saved successfully!


In [ ]:
%%writefile /content/README.md
# CareLoop Clinics — No-Show Guard

**GitHub Repository:** https://github.com/dsushrut0203/careloop-clinics-no-show-guard.git

## Project Summary

- **Part 1 — Python & Data Foundations:** Cleaned the original 1,020 appointment records to produce `appointments_clean.csv` containing 990 rows and 16 columns.
- **Part 2 — SQL Analysis:** Analysed 990 appointments across 8 clinics; the no-show rate reached 60.5% for bookings made 31+ days ahead, compared with 16.2% for bookings made 1–7 days ahead.
- **Part 3 — Machine Learning:** Selected Logistic Regression, which achieved the highest mean cross-validation ROC-AUC of 0.774 and a test ROC-AUC of 0.789.
- **Part 4 — GenAI Analysis:** Analysed 10 follow-up notes: 9 valid responses and 1 skipped blank note. Both forgotten/no-reminder notes were associated with appointments where no reminder was recorded.

## Business Recommendation

CareLoop Clinics should introduce a risk-based appointment follow-up process that prioritises patients most likely to miss their appointments. The cleaned dataset contains 990 appointments across 8 clinics, including 218 no-shows. SQL analysis identified a 60.5% no-show rate among appointments booked 31 or more days in advance, compared with 16.2% among appointments booked 1–7 days ahead. Logistic Regression achieved a test ROC-AUC of 0.789, with a classification threshold of 0.35, recall of 0.571 and precision of 0.571; it flagged 49 of the 198 test appointments for follow-up. In the GenAI analysis, 5 of the 10 reviewed appointments had no reminder recorded, and both notes classified as forgotten appointments were linked to appointments without a recorded reminder. CareLoop should prioritise reminder verification, accurate patient contact details and targeted calls for higher-risk appointments. These recommendations should be evaluated through a future pilot because the small follow-up-note sample and observational data do not establish causation.

## Selected Model and Evaluation

**Selected model: Logistic Regression**

Logistic Regression was selected because it achieved the highest mean cross-validation ROC-AUC among the evaluated models.

- Mean cross-validation ROC-AUC: 0.774
- Test ROC-AUC: 0.789
- Classification threshold: 0.35
- Test accuracy: 0.788
- Test precision: 0.571
- Test recall: 0.571
- Test F1-score: 0.571
- Test appointments flagged: 49 out of 198

The threshold of 0.35 was selected to improve the identification of potential no-shows while balancing the number of appointments requiring follow-up.

## Limitations

1. The 10 follow-up notes are an illustration and cannot show the actual reasons for missed appointments across all clinics.
2. The train/test split was random rather than past-to-future, so the evaluation may not fully represent performance on future appointments.
3. The data are observational, so associations between reminders and no-shows do not prove that reminders cause a reduction in missed appointments.

## Conclusion

The project combines data cleaning, SQL analysis, machine learning and GenAI-assisted follow-up-note classification to help CareLoop Clinics understand appointment no-shows and identify appointments requiring attention. A risk-based follow-up and reminder-verification process is recommended for further evaluation.

Writing /content/README.md


In [ ]:
import os
print(os.path.exists("/content/README.md"))

True


In [ ]:
import os

path = "/content/load_data.sql"

print("File exists:", os.path.exists(path))

if os.path.exists(path):
    print("File size:", os.path.getsize(path), "bytes")

File exists: True
File size: 156087 bytes


In [ ]:
path = "/content/load_data.sql"

with open(path, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

insert_lines = [
    line for line in lines
    if line.upper().startswith("INSERT INTO")
]

clinic_lines = [
    line for line in insert_lines
    if line.upper().startswith("INSERT INTO CLINICS")
]

appointment_lines = [
    line for line in insert_lines
    if line.upper().startswith("INSERT INTO APPOINTMENTS")
]

print("Total INSERT statements:", len(insert_lines))
print("Clinic INSERT statements:", len(clinic_lines))
print("Appointment INSERT statements:", len(appointment_lines))
print("First statement:", insert_lines[0] if insert_lines else "None")
print("Last statement:", insert_lines[-1] if insert_lines else "None")

Total INSERT statements: 998
Clinic INSERT statements: 8
Appointment INSERT statements: 990
First statement: INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 19.2);
Last statement: INSERT INTO appointments VALUES ('A1000', 'P085', 'C08', '2024-12-03', '2025-01-17', 38, 'M', 10.8, 550, 'App', 'Follow-up', 'Yes', 2, 'No-Show', 45, 1);


In [ ]:
import os

required_files = [
    "appointments_clean.csv",
    "clinics_enriched.csv",
    "load_data.sql",
    "test_risk_scores.csv",
    "notes_analysed.csv",
    "README.md"
]

for name in required_files:
    path = "/content/" + name
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(f"{status:8} {name}")

FOUND    appointments_clean.csv
FOUND    clinics_enriched.csv
FOUND    load_data.sql
FOUND    test_risk_scores.csv
FOUND    notes_analysed.csv
FOUND    README.md
